In [9]:
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance


from google.colab import files

uploaded = files.upload()

uploaded_files = list(uploaded.keys())

if len(uploaded_files) == 0:
    raise FileNotFoundError("No file was uploaded.")

print("Uploaded files:", uploaded_files)

DATA_PATH = None
for f in uploaded_files:
    if f.lower().endswith(".zip") or f.lower().endswith(".csv"):
        DATA_PATH = f
        break

if DATA_PATH is None:
    raise FileNotFoundError("Please upload either a .csv file or the Kaggle .zip file.")

print("Using file:", DATA_PATH)

OUTPUT_DIR = Path("amazon_delivery_project_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

def load_data(path: str) -> pd.DataFrame:
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as zf:
            csv_name = None
            for name in zf.namelist():
                if name.lower().endswith(".csv"):
                    csv_name = name
                    break
            if csv_name is None:
                raise FileNotFoundError("No CSV file found inside the ZIP file.")
            with zf.open(csv_name) as f:
                df = pd.read_csv(f)
    else:
        df = pd.read_csv(path)
    return df

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return r * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def savefig(name: str):
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=220, bbox_inches="tight")
    plt.close()

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

df = load_data(DATA_PATH)
original_rows = len(df)

print("Original dataset shape:", df.shape)
print(df.head())

for col in ["Weather", "Traffic", "Vehicle", "Area", "Category", "Order_Time", "Pickup_Time", "Order_Date"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df = df.replace({"nan": np.nan, "NaN": np.nan, "NaN ": np.nan, "": np.nan})

df["Order_Date_dt"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Order_Time_dt"] = pd.to_datetime(df["Order_Time"], format="%H:%M:%S", errors="coerce")
df["Pickup_Time_dt"] = pd.to_datetime(df["Pickup_Time"], format="%H:%M:%S", errors="coerce")

order_mins = df["Order_Time_dt"].dt.hour * 60 + df["Order_Time_dt"].dt.minute
pickup_mins = df["Pickup_Time_dt"].dt.hour * 60 + df["Pickup_Time_dt"].dt.minute
df["Pickup_Delay_Min"] = (pickup_mins - order_mins) % (24 * 60)

df["Distance_KM"] = haversine_km(
    df["Store_Latitude"],
    df["Store_Longitude"],
    df["Drop_Latitude"],
    df["Drop_Longitude"],
)

geo_outliers = int((df["Distance_KM"] > 100).sum())
df = df[df["Distance_KM"] <= 100].copy()

print("Rows removed as geographic outliers:", geo_outliers)
print("Rows after cleaning:", len(df))

df["Order_Hour"] = df["Order_Time_dt"].dt.hour
df["Order_Minute"] = df["Order_Time_dt"].dt.minute
df["Weekday"] = df["Order_Date_dt"].dt.day_name()
df["Month"] = df["Order_Date_dt"].dt.month
df["Is_Weekend"] = (df["Order_Date_dt"].dt.dayofweek >= 5).astype(int)
df["Peak_Lunch"] = df["Order_Hour"].between(12, 14).astype(int)
df["Peak_Dinner"] = df["Order_Hour"].between(18, 21).astype(int)
df["Fast_120"] = (df["Delivery_Time"] <= 120).astype(int)
df["Standard_150"] = (df["Delivery_Time"] <= 150).astype(int)
df["Long_Order"] = (df["Distance_KM"] > df["Distance_KM"].quantile(0.75)).astype(int)

traffic_num = df["Traffic"].map({"Low": 1, "Medium": 2, "High": 3, "Jam": 4})
df["Distance_x_Traffic"] = df["Distance_KM"] * traffic_num

summary_metrics = {
    "original_rows": int(original_rows),
    "rows_after_cleaning": int(len(df)),
    "geo_outliers_removed": int(geo_outliers),
    "average_delivery_time_min": round(float(df["Delivery_Time"].mean()), 2),
    "median_delivery_time_min": round(float(df["Delivery_Time"].median()), 2),
    "q1_delivery_time_min": round(float(df["Delivery_Time"].quantile(0.25)), 2),
    "q3_delivery_time_min": round(float(df["Delivery_Time"].quantile(0.75)), 2),
    "p90_delivery_time_min": round(float(df["Delivery_Time"].quantile(0.90)), 2),
    "p95_delivery_time_min": round(float(df["Delivery_Time"].quantile(0.95)), 2),
    "average_distance_km": round(float(df["Distance_KM"].mean()), 2),
    "median_distance_km": round(float(df["Distance_KM"].median()), 2),
    "service_level_120_pct": round(float((df["Delivery_Time"] <= 120).mean() * 100), 2),
    "service_level_150_pct": round(float((df["Delivery_Time"] <= 150).mean() * 100), 2),
    "service_level_180_pct": round(float((df["Delivery_Time"] <= 180).mean() * 100), 2),
}

traffic_summary = df.groupby("Traffic")["Delivery_Time"].agg(["mean", "median", "count"]).round(2).sort_values("mean")
vehicle_summary = df.groupby("Vehicle")["Delivery_Time"].agg(["mean", "median", "count"]).round(2).sort_values("mean")
area_summary = df.groupby("Area")["Delivery_Time"].agg(["mean", "median", "count"]).round(2).sort_values("mean")
weather_summary = df.groupby("Weather")["Delivery_Time"].agg(["mean", "median", "count"]).round(2).sort_values("mean")
pickup_delay_summary = df.groupby("Pickup_Delay_Min")["Delivery_Time"].mean().round(2).reset_index(name="Mean_Delivery_Time")

print("\nSummary Metrics:")
for k, v in summary_metrics.items():
    print(f"{k}: {v}")

plt.figure(figsize=(8, 5))
plt.hist(df["Delivery_Time"], bins=30)
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Number of Orders")
plt.title("Distribution of Delivery Time")
savefig("01_delivery_time_distribution.png")

traffic_order = [x for x in ["Low", "Medium", "High", "Jam"] if x in df["Traffic"].dropna().unique()]
plt.figure(figsize=(8, 5))
plt.boxplot([df.loc[df["Traffic"] == x, "Delivery_Time"] for x in traffic_order], tick_labels=traffic_order)
plt.xlabel("Traffic Condition")
plt.ylabel("Delivery Time (minutes)")
plt.title("Delivery Time by Traffic")
savefig("02_delivery_by_traffic.png")

plt.figure(figsize=(8, 5))
plt.scatter(df["Distance_KM"], df["Delivery_Time"], s=8)
plt.xlabel("Distance (km)")
plt.ylabel("Delivery Time (minutes)")
plt.title("Distance vs Delivery Time")
savefig("03_distance_vs_delivery.png")

hourly = df.groupby("Order_Hour")["Delivery_Time"].mean().reset_index()
plt.figure(figsize=(8, 5))
plt.plot(hourly["Order_Hour"], hourly["Delivery_Time"], marker="o")
plt.xlabel("Order Hour")
plt.ylabel("Average Delivery Time (minutes)")
plt.title("Average Delivery Time by Order Hour")
plt.xticks(range(0, 24, 2))
savefig("04_average_delivery_by_hour.png")

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_avg = df.groupby("Weekday")["Delivery_Time"].mean().reindex(weekday_order).dropna()
plt.figure(figsize=(8, 5))
plt.bar(weekday_avg.index, weekday_avg.values)
plt.xlabel("Weekday")
plt.ylabel("Average Delivery Time (minutes)")
plt.title("Average Delivery Time by Weekday")
plt.xticks(rotation=30)
savefig("05_average_delivery_by_weekday.png")

vehicle_order = sorted(df["Vehicle"].dropna().unique())
plt.figure(figsize=(8, 5))
plt.boxplot([df.loc[df["Vehicle"] == x, "Delivery_Time"] for x in vehicle_order], tick_labels=vehicle_order)
plt.xlabel("Vehicle")
plt.ylabel("Delivery Time (minutes)")
plt.title("Delivery Time by Vehicle Type")
savefig("06_delivery_by_vehicle.png")

area_order = sorted(df["Area"].dropna().unique())
plt.figure(figsize=(8, 5))
plt.boxplot([df.loc[df["Area"] == x, "Delivery_Time"] for x in area_order], tick_labels=area_order)
plt.xlabel("Area Type")
plt.ylabel("Delivery Time (minutes)")
plt.title("Delivery Time by Area Type")
plt.xticks(rotation=20)
savefig("07_delivery_by_area.png")

plt.figure(figsize=(7, 4.5))
plt.bar(pickup_delay_summary["Pickup_Delay_Min"].astype(str), pickup_delay_summary["Mean_Delivery_Time"])
plt.xlabel("Pickup Delay (minutes)")
plt.ylabel("Average Delivery Time (minutes)")
plt.title("Effect of Pickup Delay")
savefig("08_pickup_delay_effect.png")

pivot = df.pivot_table(index="Traffic", columns="Order_Hour", values="Delivery_Time", aggfunc="mean").reindex(index=traffic_order)
plt.figure(figsize=(10, 4.8))
plt.imshow(pivot, aspect="auto")
plt.colorbar(label="Average Delivery Time (minutes)")
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel("Order Hour")
plt.ylabel("Traffic")
plt.title("Heatmap: Delivery Time by Traffic and Order Hour")
savefig("09_traffic_hour_heatmap.png")

daily = df.groupby("Order_Date_dt")["Delivery_Time"].mean().sort_index()
center = daily.mean()
std = daily.std()
plt.figure(figsize=(9, 4.8))
plt.plot(daily.index, daily.values, marker="o")
plt.axhline(center)
plt.axhline(center + 3 * std, linestyle="--")
plt.axhline(max(center - 3 * std, 0), linestyle="--")
plt.xlabel("Date")
plt.ylabel("Average Delivery Time (minutes)")
plt.title("Daily Mean Delivery Time Control Chart")
plt.xticks(rotation=30)
savefig("10_control_chart.png")

features = [
    "Agent_Age",
    "Agent_Rating",
    "Distance_KM",
    "Pickup_Delay_Min",
    "Order_Hour",
    "Order_Minute",
    "Month",
    "Is_Weekend",
    "Peak_Lunch",
    "Peak_Dinner",
    "Distance_x_Traffic",
    "Weather",
    "Traffic",
    "Vehicle",
    "Area",
    "Category",
    "Weekday",
]

X = df[features].copy()
y = df["Delivery_Time"].copy()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline = DummyRegressor(strategy="median")

linear = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ])),
    ("model", LinearRegression()),
])

boost = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median"))
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols),
    ])),
    ("model", HistGradientBoostingRegressor(
        max_iter=120,
        max_depth=8,
        learning_rate=0.05,
        random_state=42
    )),
])

baseline.fit(np.zeros((len(y_train), 1)), y_train)
pred_base = baseline.predict(np.zeros((len(y_test), 1)))

linear.fit(X_train, y_train)
pred_lin = linear.predict(X_test)

boost.fit(X_train, y_train)
pred_boost = boost.predict(X_test)

model_results = pd.DataFrame([
    {
        "Model": "Baseline Median",
        "MAE": mean_absolute_error(y_test, pred_base),
        "RMSE": rmse(y_test, pred_base),
        "R2": r2_score(y_test, pred_base)
    },
    {
        "Model": "Linear Regression",
        "MAE": mean_absolute_error(y_test, pred_lin),
        "RMSE": rmse(y_test, pred_lin),
        "R2": r2_score(y_test, pred_lin)
    },
    {
        "Model": "HistGradientBoosting",
        "MAE": mean_absolute_error(y_test, pred_boost),
        "RMSE": rmse(y_test, pred_boost),
        "R2": r2_score(y_test, pred_boost)
    },
]).round(4).sort_values("RMSE")

print("\nModel Results:")
print(model_results)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for name, model in [
    ("Linear Regression", linear),
    ("HistGradientBoosting", boost)
]:
    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=1,
        return_train_score=False
    )

    cv_rows.append({
        "Model": name,
        "CV_MAE_Mean": -cv_scores["test_mae"].mean(),
        "CV_MAE_STD": cv_scores["test_mae"].std(),
        "CV_RMSE_Mean": -cv_scores["test_rmse"].mean(),
        "CV_RMSE_STD": cv_scores["test_rmse"].std(),
        "CV_R2_Mean": cv_scores["test_r2"].mean(),
        "CV_R2_STD": cv_scores["test_r2"].std(),
    })

cv_results = pd.DataFrame(cv_rows).round(4)

print("\nCross-Validation Results:")
print(cv_results)

perm_n = min(3000, len(X_test))
perm_idx = np.random.default_rng(7).choice(len(X_test), size=perm_n, replace=False)

perm = permutation_importance(
    boost,
    X_test.iloc[perm_idx],
    y_test.iloc[perm_idx],
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42
)

top_predictors = pd.DataFrame({
    "Feature": X.columns,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False).head(10).round(4)

print("\nTop Predictors:")
print(top_predictors)

pred_eval = X_test.copy()
pred_eval["Actual"] = y_test.values
pred_eval["Predicted"] = pred_boost
pred_eval["Residual"] = pred_eval["Actual"] - pred_eval["Predicted"]
pred_eval["Absolute_Error"] = np.abs(pred_eval["Residual"])

error_by_traffic = pred_eval.groupby("Traffic")["Absolute_Error"].agg(["mean", "median", "count"]).round(2).reset_index()
error_by_traffic.columns = ["Traffic", "Mean_Absolute_Error", "Median_Absolute_Error", "Orders"]

segment_summary = df.groupby(["Traffic", "Weather"]).agg(
    Orders=("Delivery_Time", "size"),
    Mean_Delivery_Time=("Delivery_Time", "mean"),
    Median_Delivery_Time=("Delivery_Time", "median"),
    Service_Level_120=("Fast_120", "mean"),
    Service_Level_150=("Standard_150", "mean")
).reset_index()

segment_summary["Mean_Delivery_Time"] = segment_summary["Mean_Delivery_Time"].round(2)
segment_summary["Median_Delivery_Time"] = segment_summary["Median_Delivery_Time"].round(2)
segment_summary["Service_Level_120"] = (segment_summary["Service_Level_120"] * 100).round(2)
segment_summary["Service_Level_150"] = (segment_summary["Service_Level_150"] * 100).round(2)
segment_summary = segment_summary.sort_values(["Mean_Delivery_Time", "Orders"], ascending=[False, False])

print("\nSegment Summary (top 15 rows):")
print(segment_summary.head(15))

baseline_row = model_results.loc[model_results["Model"] == "Baseline Median"].iloc[0]
linear_row = model_results.loc[model_results["Model"] == "Linear Regression"].iloc[0]
boost_row = model_results.loc[model_results["Model"] == "HistGradientBoosting"].iloc[0]

summary_metrics["best_model"] = "HistGradientBoosting"
summary_metrics["best_model_mae"] = round(float(boost_row["MAE"]), 4)
summary_metrics["best_model_rmse"] = round(float(boost_row["RMSE"]), 4)
summary_metrics["best_model_r2"] = round(float(boost_row["R2"]), 4)

summary_metrics["boost_vs_baseline_mae_reduction_pct"] = round(float((baseline_row["MAE"] - boost_row["MAE"]) / baseline_row["MAE"] * 100), 2)
summary_metrics["boost_vs_baseline_rmse_reduction_pct"] = round(float((baseline_row["RMSE"] - boost_row["RMSE"]) / baseline_row["RMSE"] * 100), 2)
summary_metrics["boost_vs_linear_mae_reduction_pct"] = round(float((linear_row["MAE"] - boost_row["MAE"]) / linear_row["MAE"] * 100), 2)
summary_metrics["boost_vs_linear_r2_gain"] = round(float(boost_row["R2"] - linear_row["R2"]), 4)

boost_cv_row = cv_results.loc[cv_results["Model"] == "HistGradientBoosting"].iloc[0]
summary_metrics["cv_mae_mean"] = round(float(boost_cv_row["CV_MAE_Mean"]), 4)
summary_metrics["cv_mae_std"] = round(float(boost_cv_row["CV_MAE_STD"]), 4)
summary_metrics["cv_rmse_mean"] = round(float(boost_cv_row["CV_RMSE_Mean"]), 4)
summary_metrics["cv_rmse_std"] = round(float(boost_cv_row["CV_RMSE_STD"]), 4)
summary_metrics["cv_r2_mean"] = round(float(boost_cv_row["CV_R2_Mean"]), 4)
summary_metrics["cv_r2_std"] = round(float(boost_cv_row["CV_R2_STD"]), 4)


plt.figure(figsize=(7, 4.5))
plt.bar(model_results["Model"], model_results["MAE"])
plt.xlabel("Model")
plt.ylabel("MAE (minutes)")
plt.title("Model Comparison by MAE")
plt.xticks(rotation=20)
savefig("11_model_comparison.png")

plt.figure(figsize=(7, 4.5))
plt.bar(top_predictors["Feature"][::-1], top_predictors["Importance"][::-1])
plt.xlabel("Permutation Importance")
plt.ylabel("Feature")
plt.title("Top Predictors of Delivery Time")
savefig("12_top_predictors.png")


plt.figure(figsize=(7, 4.5))
plt.bar(error_by_traffic["Traffic"], error_by_traffic["Mean_Absolute_Error"])
plt.xlabel("Traffic")
plt.ylabel("Mean Absolute Error (minutes)")
plt.title("Prediction Error by Traffic Level")
savefig("13_error_by_traffic.png")


lims_min = min(pred_eval["Actual"].min(), pred_eval["Predicted"].min())
lims_max = max(pred_eval["Actual"].max(), pred_eval["Predicted"].max())

plt.figure(figsize=(6.5, 5))
plt.scatter(pred_eval["Actual"], pred_eval["Predicted"], s=8, alpha=0.35)
plt.plot([lims_min, lims_max], [lims_min, lims_max], linestyle="--")
plt.xlabel("Actual Delivery Time (minutes)")
plt.ylabel("Predicted Delivery Time (minutes)")
plt.title("Actual vs Predicted Delivery Time")
savefig("14_actual_vs_predicted.png")

plt.figure(figsize=(6.5, 4.5))
plt.hist(pred_eval["Residual"], bins=30)
plt.xlabel("Residual (Actual - Predicted)")
plt.ylabel("Number of Orders")
plt.title("Residual Distribution for HistGradientBoosting")
savefig("15_residual_distribution.png")


sla_by_traffic = df.groupby("Traffic").agg(
    Orders=("Delivery_Time", "size"),
    Mean_Delivery_Time=("Delivery_Time", "mean"),
    Service_Level_120=("Fast_120", "mean"),
    Service_Level_150=("Standard_150", "mean"),
    Service_Level_180=("Delivery_Time", lambda s: (s <= 180).mean())
).reset_index()

sla_by_weather = df.groupby("Weather").agg(
    Orders=("Delivery_Time", "size"),
    Mean_Delivery_Time=("Delivery_Time", "mean"),
    Service_Level_120=("Fast_120", "mean"),
    Service_Level_150=("Standard_150", "mean"),
    Service_Level_180=("Delivery_Time", lambda s: (s <= 180).mean())
).reset_index()

for table in [sla_by_traffic, sla_by_weather]:
    table["Mean_Delivery_Time"] = table["Mean_Delivery_Time"].round(2)
    table["Service_Level_120"] = (table["Service_Level_120"] * 100).round(2)
    table["Service_Level_150"] = (table["Service_Level_150"] * 100).round(2)
    table["Service_Level_180"] = (table["Service_Level_180"] * 100).round(2)

sla_by_traffic = sla_by_traffic.sort_values("Mean_Delivery_Time", ascending=False)
sla_by_weather = sla_by_weather.sort_values("Mean_Delivery_Time", ascending=False)

worst_segments = df.groupby(["Weather", "Traffic", "Area"]).agg(
    Orders=("Delivery_Time", "size"),
    Mean_Delivery_Time=("Delivery_Time", "mean"),
    Median_Delivery_Time=("Delivery_Time", "median"),
    Service_Level_150=("Standard_150", "mean")
).reset_index()

worst_segments = worst_segments[worst_segments["Orders"] >= 100].copy()
worst_segments["Mean_Delivery_Time"] = worst_segments["Mean_Delivery_Time"].round(2)
worst_segments["Median_Delivery_Time"] = worst_segments["Median_Delivery_Time"].round(2)
worst_segments["Service_Level_150"] = (worst_segments["Service_Level_150"] * 100).round(2)
worst_segments = worst_segments.sort_values(["Mean_Delivery_Time", "Orders"], ascending=[False, False])

print("\nService Level by Traffic:")
print(sla_by_traffic.to_string(index=False))

print("\nService Level by Weather:")
print(sla_by_weather.to_string(index=False))

print("\nWorst High-Delay Segments:")
print(worst_segments.head(10).to_string(index=False))

plt.figure(figsize=(7, 4.5))
plt.bar(sla_by_traffic["Traffic"], sla_by_traffic["Service_Level_150"])
plt.xlabel("Traffic")
plt.ylabel("Service Level within 150 Minutes (%)")
plt.title("Service-Level Attainment by Traffic")
savefig("16_service_level_150_by_traffic.png")


top_segments_plot = worst_segments.head(8).copy()
top_segments_plot["Segment"] = (
    top_segments_plot["Weather"].astype(str)
    + " | "
    + top_segments_plot["Traffic"].astype(str)
    + " | "
    + top_segments_plot["Area"].astype(str)
)

plt.figure(figsize=(8.5, 5.5))
plt.barh(top_segments_plot["Segment"][::-1], top_segments_plot["Mean_Delivery_Time"][::-1])
plt.xlabel("Mean Delivery Time (minutes)")
plt.ylabel("Segment")
plt.title("Highest-Delay Delivery Segments")
savefig("17_highest_delay_segments.png")


cv_plot = cv_results.copy()
plt.figure(figsize=(7, 4.5))
plt.bar(cv_plot["Model"], cv_plot["CV_RMSE_Mean"])
plt.xlabel("Model")
plt.ylabel("Cross-Validated RMSE (minutes)")
plt.title("Cross-Validated RMSE by Model")
plt.xticks(rotation=15)
savefig("18_cv_rmse_comparison.png")

pilot_kpis = pd.DataFrame([
    {
        "KPI": "Service Level <= 150 Minutes",
        "Current_Baseline": summary_metrics["service_level_150_pct"],
        "Why_It_Matters": "Directly reflects promise-window attainment"
    },
    {
        "KPI": "90th Percentile Delivery Time",
        "Current_Baseline": summary_metrics["p90_delivery_time_min"],
        "Why_It_Matters": "Captures tail-risk performance"
    },
    {
        "KPI": "95th Percentile Delivery Time",
        "Current_Baseline": summary_metrics["p95_delivery_time_min"],
        "Why_It_Matters": "Captures extreme delays"
    },
    {
        "KPI": "Model MAE",
        "Current_Baseline": summary_metrics["best_model_mae"],
        "Why_It_Matters": "Measures prediction accuracy for risk screening"
    }
])

print("\nPilot KPI Table:")
print(pilot_kpis.to_string(index=False))

sla_by_traffic.to_csv(OUTPUT_DIR / "sla_by_traffic.csv", index=False)
sla_by_weather.to_csv(OUTPUT_DIR / "sla_by_weather.csv", index=False)
worst_segments.to_csv(OUTPUT_DIR / "worst_segments.csv", index=False)
pilot_kpis.to_csv(OUTPUT_DIR / "pilot_kpis.csv", index=False)

(OUTPUT_DIR / "summary_metrics.json").write_text(json.dumps(summary_metrics, indent=2))
traffic_summary.to_csv(OUTPUT_DIR / "traffic_summary.csv")
vehicle_summary.to_csv(OUTPUT_DIR / "vehicle_summary.csv")
area_summary.to_csv(OUTPUT_DIR / "area_summary.csv")
weather_summary.to_csv(OUTPUT_DIR / "weather_summary.csv")
pickup_delay_summary.to_csv(OUTPUT_DIR / "pickup_delay_summary.csv", index=False)
model_results.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
cv_results.to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
top_predictors.to_csv(OUTPUT_DIR / "top_predictors.csv", index=False)
error_by_traffic.to_csv(OUTPUT_DIR / "error_by_traffic.csv", index=False)
segment_summary.to_csv(OUTPUT_DIR / "segment_summary.csv", index=False)
pred_eval.head(1000).to_csv(OUTPUT_DIR / "prediction_diagnostics_sample.csv", index=False)

print("\nDone. Outputs saved to:", OUTPUT_DIR.resolve())

import shutil

zip_name = "amazon_delivery_project_outputs.zip"
shutil.make_archive("amazon_delivery_project_outputs", "zip", OUTPUT_DIR)

print("Created zip file:", zip_name)

files.download(zip_name)

Saving archive (2).zip to archive (2) (7).zip
Uploaded files: ['archive (2) (7).zip']
Using file: archive (2) (7).zip
Original dataset shape: (43739, 16)
        Order_ID  Agent_Age  Agent_Rating  Store_Latitude  Store_Longitude  \
0  ialx566343618         37           4.9       22.745049        75.892471   
1  akqg208421122         34           4.5       12.913041        77.683237   
2  njpu434582536         23           4.4       12.914264        77.678400   
3  rjto796129700         38           4.7       11.003669        76.976494   
4  zguw716275638         32           4.6       12.972793        80.249982   

   Drop_Latitude  Drop_Longitude  Order_Date Order_Time Pickup_Time  \
0      22.765049       75.912471  2022-03-19   11:30:00    11:45:00   
1      13.043041       77.813237  2022-03-25   19:45:00    19:50:00   
2      12.924264       77.688400  2022-03-19   08:30:00    08:45:00   
3      11.053669       77.026494  2022-04-05   18:00:00    18:10:00   
4      13.012793      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>